# 05 - Evaluation: comparing all four retrievers

Builds the same chunk set in BM25, dense, hybrid, and reranking retrievers,
then runs `evaluate_retriever` on a set of multi-year SEC questions and
prints the metrics side-by-side.

**Methodology note.** Two gold sets are evaluated:

1. **Hand-curated gold ids** -- chunk ids verified by inspection to contain
   the answer for each query. This is the honest evaluation.
2. **Oracle (reranker as labeler)** -- the cross-encoder reranker's top-3
   chunks are taken as ground truth. This biases the metrics in favor of
   the reranker (it scores against itself), but it's a useful upper bound
   when no human labels exist.

In [1]:
import os
import sys
import warnings

sys.path.insert(0, os.path.abspath('..'))

from rag.data_ingestion import chunk_documents, load_documents
from rag.evaluation import EvalExample, evaluate_retriever
from rag.retrievers import (
    BM25Retriever,
    DenseRetriever,
    HybridRetriever,
    RerankerRetriever,
)

warnings.filterwarnings('ignore')

FILE_PATH = '../data/google_10K.pdf'

docs = load_documents(FILE_PATH)
chunks = chunk_documents(docs, chunk_size=2000, chunk_overlap=200)
print(f'Loaded {len(docs)} pages -> {len(chunks)} chunks')

Loaded 107 pages -> 230 chunks


In [2]:
bm25 = BM25Retriever();           bm25.add_documents(chunks)
dense = DenseRetriever();         dense.add_documents(chunks)
hybrid = HybridRetriever();       hybrid.add_documents(chunks)
reranker = RerankerRetriever(HybridRetriever()); reranker.add_documents(chunks)

retrievers = {
    'bm25':     bm25,
    'dense':    dense,
    'hybrid':   hybrid,
    'reranker': reranker,
}
list(retrievers)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['bm25', 'dense', 'hybrid', 'reranker']

## Hand-curated gold set

Each entry's `relevant_doc_ids` were chosen by reading the chunks and
verifying that they contain the answer. This is the bias-free evaluation.

In [3]:
hand_labeled = [
    EvalExample(
        question='Total revenues for every fiscal year reported, with year-over-year growth.',
        relevant_doc_ids=['chunk_96', 'chunk_132'],
    ),
    EvalExample(
        question='Research and development expenses each year and as a percentage of revenue.',
        relevant_doc_ids=['chunk_101', 'chunk_132', 'chunk_94'],
    ),
    EvalExample(
        question='Operating income, operating margin, and net income for every reported fiscal year.',
        relevant_doc_ids=['chunk_96', 'chunk_103', 'chunk_132'],
    ),
    EvalExample(
        question='Cash, cash equivalents, and marketable securities balances at year end for each year.',
        relevant_doc_ids=['chunk_106', 'chunk_131'],
    ),
    EvalExample(
        question='Effective tax rate and provision for income taxes for each year.',
        relevant_doc_ids=['chunk_105', 'chunk_203', 'chunk_207'],
    ),
    EvalExample(
        question='Total number of full-time employees at year end.',
        relevant_doc_ids=['chunk_98'],
    ),
]
for ex in hand_labeled:
    print(f'{ex.question[:80]:<82} {ex.relevant_doc_ids}')

Total revenues for every fiscal year reported, with year-over-year growth.         ['chunk_96', 'chunk_132']
Research and development expenses each year and as a percentage of revenue.        ['chunk_101', 'chunk_132', 'chunk_94']
Operating income, operating margin, and net income for every reported fiscal yea   ['chunk_96', 'chunk_103', 'chunk_132']
Cash, cash equivalents, and marketable securities balances at year end for each    ['chunk_106', 'chunk_131']
Effective tax rate and provision for income taxes for each year.                   ['chunk_105', 'chunk_203', 'chunk_207']
Total number of full-time employees at year end.                                   ['chunk_98']


In [4]:
print('=== Hand-curated gold set (honest eval) ===')
for name, r in retrievers.items():
    report = evaluate_retriever(r, hand_labeled, k=5)
    metrics = '  '.join(f'{k}={v:.3f}' for k, v in report.metrics.items())
    print(f'{name:<10} {metrics}')

=== Hand-curated gold set (honest eval) ===
bm25       hit@5=0.833  precision@5=0.267  recall@5=0.500  ndcg@5=0.498  mrr=0.639  map=0.403
dense      hit@5=0.833  precision@5=0.233  recall@5=0.444  ndcg@5=0.428  mrr=0.639  map=0.320
hybrid     hit@5=0.833  precision@5=0.333  recall@5=0.611  ndcg@5=0.545  mrr=0.667  map=0.435


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


reranker   hit@5=1.000  precision@5=0.367  recall@5=0.806  ndcg@5=0.777  mrr=0.889  map=0.690


## Oracle gold set (reranker as labeler)

Useful as an upper bound on what the reranker can find, but biased: the
reranker is being graded against its own output, so its metrics are perfect
by construction. The other three retrievers are still being scored fairly
*against* the reranker.

In [5]:
oracle = []
for ex in hand_labeled:
    rerank_ids = [hit.document.metadata['doc_id'] for hit in reranker.retrieve(ex.question, k=3)]
    oracle.append(EvalExample(question=ex.question, relevant_doc_ids=rerank_ids))

print('=== Oracle gold set (reranker top-3 = ground truth) ===')
for name, r in retrievers.items():
    report = evaluate_retriever(r, oracle, k=5)
    metrics = '  '.join(f'{k}={v:.3f}' for k, v in report.metrics.items())
    print(f'{name:<10} {metrics}')

=== Oracle gold set (reranker top-3 = ground truth) ===
bm25       hit@5=1.000  precision@5=0.367  recall@5=0.611  ndcg@5=0.632  mrr=0.833  map=0.509
dense      hit@5=0.667  precision@5=0.233  recall@5=0.389  ndcg@5=0.412  mrr=0.556  map=0.333
hybrid     hit@5=1.000  precision@5=0.467  recall@5=0.778  ndcg@5=0.750  mrr=0.917  map=0.622


reranker   hit@5=1.000  precision@5=0.600  recall@5=1.000  ndcg@5=1.000  mrr=1.000  map=1.000
